In [40]:
import re
import pandas as pd
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================
INPUT_CSV = "manual_annotation_progress_saad.csv"      # must contain a column named: text
OUTPUT_CSV = "output_demographics.csv"

# ============================================================
# PATTERNS
# ============================================================

# Common age+gender formats:
# (26f), [19F], 32M, 38NB, 23f, 24yo F, etc.
AGE_GENDER_PATTERNS = [
    # Bracketed formats - high confidence
    re.compile(r'[\(\[]\s*(\d{1,3})\s*(m|f|nb|male|female|nonbinary|enby)\s*[\)\]]', re.I),
    # Standalone age+gender but NOT followed by measurement units (ml, mm, cm, kg, etc.)
    re.compile(r'\b(\d{1,3})\s*(male|female|nb|nonbinary|enby)\b(?!l|m\b|g\b)', re.I),
    # Single letter m/f only in very specific contexts (not followed by units)
    re.compile(r'\b(\d{2,3})\s*([mf])\b(?!l\b|m\b|g\b|/|\d)', re.I),
]

# Age-only formats:
# 24yo, 27 years old, at my age (26), I am 27, I'm actually 27
AGE_PATTERNS = [
    re.compile(r'\b(\d{1,3})\s*(yo|y/o|years?\s*old|yrs?\s*old)\b', re.I),
    re.compile(r'at my age\s*[\(\[]?\s*(\d{1,3})\s*[\)\]]?', re.I),
    re.compile(r'\b(?:i am|i\'m|im)\s+(?:actually\s+)?(\d{1,3})\b', re.I),
    re.compile(r'\b(\d{1,3})\s*-\s*(\d{1,3})\s*year old\b', re.I),  # age range
    re.compile(r'\b(\d{1,3})\s*-\s*(\d{1,3})\s*years old\b', re.I),
]

# Gender-only formats
GENDER_PATTERNS = [
    re.compile(r'\b(?:i am|i\'m|im)\s+(?:a\s+)?(girl|boy|man|woman|male|female|nonbinary|nb|enby)\b', re.I),
    re.compile(r'\bbiologically\s+(male|female)\b', re.I),
    re.compile(r'\b(?:a\s+)?(detrans woman|detrans man|trans woman|trans man)\b', re.I),
]

# Education level patterns - more context-aware
EDUCATION_PATTERNS = [
    # High school - keep as is
    re.compile(r'\b(high school|secondary school)\b', re.I),
    # HS only when clearly education context (in school, attending hs, etc.)
    re.compile(r'\b(?:in|at|attend|attending|finished)\s+(?:a\s+)?(hs)\b', re.I),
    
    # College/undergrad - expanded forms are safer
    re.compile(r'\b(college student|undergrad|undergraduate|bachelor[\'']s?\s+degree)\b', re.I),
    # BA/BS only when followed by degree or in clear education context
    re.compile(r'\b(ba|bs|b\.a\.|b\.s\.)\s+(?:degree|in|from|student)', re.I),
    
    # Master's - expanded form safer
    re.compile(r'\b(master[\'']s?\s+degree|m\.s\.\s+degree|m\.a\.\s+degree)\b', re.I),
    # MS/MA only in clear degree context
    re.compile(r'\b(ms|ma)\s+(?:degree|in|from|student)', re.I),
    
    # PhD and grad - these are usually safe
    re.compile(r'\b(phd|ph\.d\.|doctorate|doctoral student|grad student|graduate student)\b', re.I),
    
    # Academic roles
    re.compile(r'\b(professor|postdoc|researcher)\b', re.I),
    
    # Student year levels - require "I am/I'm" or "as a" context
    re.compile(r'\b(?:i am|i\'m|im|as\s+a)\s+(?:a\s+)?(freshman|sophomore|junior|senior)\b', re.I),
]

# ============================================================
# NORMALIZATION
# ============================================================

def normalize_gender(value: str):
    if not value:
        return None
    v = value.strip().lower()

    mapping = {
        "m": "male",
        "male": "male",
        "man": "male",
        "boy": "male",
        "f": "female",
        "female": "female",
        "woman": "female",
        "girl": "female",
        "nb": "nonbinary",
        "nonbinary": "nonbinary",
        "enby": "nonbinary",
        "trans man": "trans_man",
        "trans woman": "trans_woman",
        "detrans man": "detrans_man",
        "detrans woman": "detrans_woman",
    }
    return mapping.get(v, v)

def normalize_education(value: str):
    if not value:
        return None
    v = value.strip().lower()

    if v in {"high school", "hs", "secondary school"}:
        return "high_school"
    if v in {"college student", "undergrad", "undergraduate", "bachelor's", "bachelors", "bachelor's degree", "ba", "bs", "b.a.", "b.s."}:
        return "undergraduate"
    if v in {"master's", "masters", "master's degree", "ms", "m.s.", "ma", "m.a."}:
        return "masters"
    if v in {"phd", "ph.d.", "doctorate", "doctoral student", "grad student", "graduate student"}:
        return "phd_or_grad"
    if v in {"professor", "postdoc", "researcher"}:
        return "academic_staff"
    if v in {"freshman", "sophomore", "junior", "senior"}:
        return "student"
    return v

def is_valid_age(age):
    try:
        age = int(age)
        return 10 <= age <= 100
    except:
        return False

# ============================================================
# EXTRACTION
# ============================================================

def extract_demographics(text):
    if pd.isna(text):
        text = ""
    text = str(text)

    result = {
        "age": None,
        "age_min": None,
        "age_max": None,
        "gender": None,
        "education_level": None,
        "matched_excerpt": None,
        "pattern_type": None,
        "confidence": "low",
    }

    # --------------------------------------------------------
    # 1. Age + gender together (highest confidence)
    # --------------------------------------------------------
    for pat in AGE_GENDER_PATTERNS:
        m = pat.search(text)
        if m:
            age = m.group(1)
            gender = m.group(2)

            if is_valid_age(age):
                result["age"] = int(age)
                result["gender"] = normalize_gender(gender)
                result["matched_excerpt"] = m.group(0)
                result["pattern_type"] = "age_gender"
                result["confidence"] = "high"
                return result

    # --------------------------------------------------------
    # 2. Age range
    # --------------------------------------------------------
    for pat in AGE_PATTERNS[-2:]:
        m = pat.search(text)
        if m:
            a1, a2 = m.group(1), m.group(2)
            if is_valid_age(a1) and is_valid_age(a2):
                result["age_min"] = int(a1)
                result["age_max"] = int(a2)
                result["matched_excerpt"] = m.group(0)
                result["pattern_type"] = "age_range"
                result["confidence"] = "medium"

                # optional gender / education lookup too
                for gpat in GENDER_PATTERNS:
                    gm = gpat.search(text)
                    if gm:
                        result["gender"] = normalize_gender(gm.group(1))
                        break

                for epat in EDUCATION_PATTERNS:
                    em = epat.search(text)
                    if em:
                        result["education_level"] = normalize_education(em.group(1))
                        break

                return result

    # --------------------------------------------------------
    # 3. Age only
    # --------------------------------------------------------
    for pat in AGE_PATTERNS[:-2]:
        m = pat.search(text)
        if m:
            age = m.group(1)
            if is_valid_age(age):
                result["age"] = int(age)
                result["matched_excerpt"] = m.group(0)
                result["pattern_type"] = "age_only"
                result["confidence"] = "medium"
                break

    # --------------------------------------------------------
    # 4. Gender only
    # --------------------------------------------------------
    for pat in GENDER_PATTERNS:
        m = pat.search(text)
        if m:
            result["gender"] = normalize_gender(m.group(1))
            if result["matched_excerpt"] is None:
                result["matched_excerpt"] = m.group(0)
                result["pattern_type"] = "gender_only"
            result["confidence"] = "medium" if result["age"] is None else result["confidence"]
            break

    # --------------------------------------------------------
    # 5. Education only
    # --------------------------------------------------------
    for pat in EDUCATION_PATTERNS:
        m = pat.search(text)
        if m:
            result["education_level"] = normalize_education(m.group(1))
            if result["matched_excerpt"] is None:
                result["matched_excerpt"] = m.group(0)
                result["pattern_type"] = "education_only"
            if result["age"] is None and result["gender"] is None:
                result["confidence"] = "medium"
            break

    return result

# ============================================================
# APPLY TO DATA
# ============================================================

df = pd.read_csv(INPUT_CSV)
print(df.head())
if "text" not in df.columns:
    raise ValueError("Input CSV must contain a 'text' column.")

extracted = df["text"].apply(extract_demographics).apply(pd.Series)
out = pd.concat([df, extracted], axis=1)

out.to_csv(OUTPUT_CSV, index=False)

# Quick stats
print("Done.")
print(f"Rows: {len(out)}")
print(f"Age found: {(out['age'].notna() | out['age_min'].notna()).sum()}")
print(f"Gender found: {out['gender'].notna().sum()}")
print(f"Education found: {out['education_level'].notna().sum()}")
print(f"Saved to: {OUTPUT_CSV}")

# Show sample matches
sample = out.loc[
    out["matched_excerpt"].notna(),
    ["text", "age", "age_min", "age_max", "gender", "education_level", "matched_excerpt", "pattern_type", "confidence"]
].head(20)

print("\nSample extracted rows:")
print(sample.to_string(index=False))



SyntaxError: closing parenthesis ']' does not match opening parenthesis '(' (4045230423.py, line 51)

In [39]:
out[(out["annotation"] == 1) & (out["matched_excerpt"].notna())][[ "matched_excerpt"]]

,matched_excerpt
121,MA
169,I am 63
225,Junior
249,ba
283,Ba
289,63m
